<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/Copy_of_langsmith_evaluation_cohort11_gpt5_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 LangSmith Evaluation — Recipe & FDA Safety Bot
### Cohort 11 · GPT-5.6 · Correctness + Conciseness

This notebook evaluates a food-safety Q&A bot using **GPT-5.6** as both the target model and the LLM-as-a-Judge evaluator.

**Key fix:** GPT-5.6 does not support `temperature=0`. We omit it entirely (uses the default `temperature=1`).

In [ ]:
# ========================================
# 1. Install dependencies
# ========================================
!pip install -q \
  python-dotenv \
  langsmith \
  langchain \
  langchain-openai \
  langchain-community \
  langchain-text-splitters \
  openai \
  pandas \
  tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# ========================================
# 2. Load API keys from Google Colab Secrets
# ========================================
import os
from google.colab import userdata

OPENAI_API_KEY = userdata.get("openai")
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")

if not OPENAI_API_KEY or not LANGSMITH_API_KEY:
    raise ValueError(
        "Set 'openai' and 'LANGSMITH_API_KEY' in Colab Secrets "
        "(key icon in left sidebar)."
    )

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"

print("API keys loaded successfully.")

API keys loaded successfully.


In [ ]:
# ========================================
# 3. Create LangSmith client & dataset
# ========================================
from langsmith import Client

client = Client()

dataset_name = "Recipe Bot — FDA Safety Q/A — Cohort 11"
dataset = client.create_dataset(dataset_name)
print(f"Dataset created: {dataset_name}")

Dataset created: Recipe Bot — FDA Safety Q/A — Cohort 11


In [ ]:
# ========================================
# 4. Define test examples (34 total)
# ========================================

original_examples = [
    {
        "inputs": {
            "question": "How many teaspoons are in one tablespoon?",
            "context": "US kitchen measurement equivalents."
        },
        "outputs": {"answer": "3"}
    },
    {
        "inputs": {
            "question": "What is the safe internal temperature for cooked chicken (°C)?",
            "context": "Food safety guideline for poultry doneness."
        },
        "outputs": {"answer": "74°C"}
    },
    {
        "inputs": {
            "question": "Convert 2 US cups to milliliters.",
            "context": "Use the US legal cup for home cooking."
        },
        "outputs": {"answer": "480 ml"}
    },
    {
        "inputs": {
            "question": "What is the classic vinaigrette oil-to-acid ratio?",
            "context": "Standard salad dressing ratio."
        },
        "outputs": {"answer": "3:1"}
    },
    {
        "inputs": {
            "question": "Substitute for 1 cup light brown sugar using white sugar and molasses.",
            "context": "Common home-baking substitution."
        },
        "outputs": {"answer": "1 cup white sugar + 1 tbsp molasses"}
    },
    {
        "inputs": {
            "question": "Minimum internal temperature for medium-rare steak (°C).",
            "context": "Typical doneness temperature."
        },
        "outputs": {"answer": "57°C"}
    },
    {
        "inputs": {
            "question": "How many grams are in 1 ounce (oz)?",
            "context": "Kitchen weight conversion."
        },
        "outputs": {"answer": "28.35 g"}
    },
    {
        "inputs": {
            "question": "What gas is produced when baking soda reacts with an acid?",
            "context": "Leavening reaction in quick breads."
        },
        "outputs": {"answer": "Carbon dioxide"}
    },
    {
        "inputs": {
            "question": "Boiling time for a soft-boiled egg (runny yolk) after simmering starts.",
            "context": "Stovetop method, large eggs."
        },
        "outputs": {"answer": "6 minutes"}
    },
    {
        "inputs": {
            "question": "How many tablespoons are in 1/4 cup (US)?",
            "context": "US kitchen measurement equivalents."
        },
        "outputs": {"answer": "4 tbsp"}
    },

]

extra_examples = [
    {
        "inputs": {
            "question": "What is the USDA safe internal temperature for ground beef (°C)?",
            "context": "US food safety temperature guidelines."
        },
        "outputs": {"answer": "71°C"}
    },
    {
        "inputs": {
            "question": "What is the bacterial 'Danger Zone' temperature range (°C)?",
            "context": "FDA food safety storage rules."
        },
        "outputs": {"answer": "5°C to 60°C"}
    },
    {
        "inputs": {
            "question": "How long can cooked food safely sit at room temperature before it should be discarded?",
            "context": "General US food safety rule for perishable foods."
        },
        "outputs": {"answer": "Maximum 2 hours"}
    },
    {
        "inputs": {
            "question": "To what internal temperature (°C) should leftovers be reheated for safety?",
            "context": "US food safety guideline for reheating leftovers."
        },
        "outputs": {"answer": "74°C"}
    },
    {
        "inputs": {
            "question": "Should raw chicken be washed before cooking?",
            "context": "FDA guidance on cross-contamination in home kitchens."
        },
        "outputs": {"answer": "No — washing raw chicken can spread bacteria through splashing."}
    },
    {
        "inputs": {
            "question": "What is the minimum recommended time for proper handwashing?",
            "context": "Food safety guidance for handwashing."
        },
        "outputs": {"answer": "At least 20 seconds"}
    },
    {
        "inputs": {
            "question": "How long can raw chicken be safely stored in the refrigerator?",
            "context": "USDA guidance for refrigerated storage of raw poultry."
        },
        "outputs": {"answer": "1 to 2 days"}
    },
    {
        "inputs": {
            "question": "At what freezer temperature (°C) should food be stored to keep it safe long-term?",
            "context": "US food safety freezer storage guidelines."
        },
        "outputs": {"answer": "−18°C or lower"}
    },
    {
        "inputs": {
            "question": "Name three allergens from the FDA Big Nine list.",
            "context": "US FDA major food allergen list."
        },
        "outputs": {"answer": "Examples include milk, eggs, and peanuts."}
    },
    {
        "inputs": {
            "question": "Is sesame one of the FDA-recognized major allergens?",
            "context": "FDA Big Nine allergen list, updated in recent years."
        },
        "outputs": {"answer": "Yes, sesame is one of the major allergens."}
    },
    {
        "inputs": {
            "question": "How many milliliters are in one US tablespoon?",
            "context": "US kitchen measurement standards."
        },
        "outputs": {"answer": "About 14.79 ml (often rounded to 15 ml)."}
    },
    {
        "inputs": {
            "question": "How many fluid ounces are in one US cup?",
            "context": "US kitchen volume measurements."
        },
        "outputs": {"answer": "8 fluid ounces"}
    },
    {
        "inputs": {
            "question": "Approximately how many grams are in 1 cup of all-purpose flour?",
            "context": "Typical baking reference for US recipes."
        },
        "outputs": {"answer": "Around 120 g, though it can vary with measurement method."}
    },
    {
        "inputs": {
            "question": "How many teaspoons are in 1/2 tablespoon?",
            "context": "US kitchen measurement equivalents."
        },
        "outputs": {"answer": "1.5 teaspoons"}
    },
    {
        "inputs": {
            "question": "Are US and UK pints the same size?",
            "context": "International measurement comparisons that can confuse recipes."
        },
        "outputs": {"answer": "No — a US pint is about 473 ml, while a UK pint is about 568 ml."}
    },
    {
        "inputs": {
            "question": "How many sticks of butter make 1 cup in US recipes?",
            "context": "Common US baking measurement for butter."
        },
        "outputs": {"answer": "2 sticks of butter equal 1 cup (about 226 g)."}
    },
    {
        "inputs": {
            "question": "Can you rely on the color of chicken meat alone to know if it is safely cooked?",
            "context": "FDA advice on checking doneness of poultry."
        },
        "outputs": {"answer": "No — color is not reliable; you must check that the internal temperature reaches 74°C."}
    },
    {
        "inputs": {
            "question": "Does freezing meat kill harmful bacteria?",
            "context": "Food preservation and safety guidance."
        },
        "outputs": {"answer": "No — freezing usually does not kill bacteria; it mainly stops them from growing."}
    },
    {
        "inputs": {
            "question": "Is 165°F the same as 65°C for cooked chicken?",
            "context": "Comparing common food safety temperatures between Fahrenheit and Celsius."
        },
        "outputs": {"answer": "No — 165°F is about 74°C, not 65°C."}
    },
    {
        "inputs": {
            "question": "Does 1 US cup of flour always weigh exactly 120 g?",
            "context": "Baking measurement variability."
        },
        "outputs": {"answer": "No — 120 g is a common reference, but actual weight can range roughly 100–130 g depending on how it is measured."}
    },
    {
        "inputs": {
            "question": "Is rare steak safe for everyone to eat?",
            "context": "Food safety risk levels for different groups of people."
        },
        "outputs": {"answer": "Rare steak can be acceptable for healthy adults when properly handled, but higher-risk groups like pregnant people, older adults, and immunocompromised individuals are advised to avoid undercooked meat."}
    },
    {
        "inputs": {
            "question": "What is the minimum hot-holding temperature (°C) recommended for cooked foods?",
            "context": "US food service guidance for keeping cooked food hot and safe."
        },
        "outputs": {"answer": "About 57°C or higher is recommended for hot holding."}
    },
    {
        "inputs": {
            "question": "Is there a single exact temperature for 'room temperature' in recipes?",
            "context": "Culinary terminology and approximate temperature ranges."
        },
        "outputs": {"answer": "No — it is not an exact standard; in cooking it usually means around 20–22°C."}
    },
]

# Upload to LangSmith
client.create_examples(
    dataset_id=dataset.id,
    examples=original_examples + extra_examples
)

total = len(original_examples) + len(extra_examples)
print(f"Uploaded {total} examples to '{dataset_name}'")

Uploaded 34 examples to 'Recipe Bot — FDA Safety Q/A — Cohort 11'


In [ ]:
# ========================================
# 5. Wrap OpenAI client for LangSmith tracing
# ========================================
import openai
from langsmith import wrappers

openai_client = wrappers.wrap_openai(openai.OpenAI())
print("OpenAI client wrapped for LangSmith tracing.")

OpenAI client wrapped for LangSmith tracing.


## 🔍 Define Evaluators (LLM-as-a-Judge)

**Critical fix:** GPT-5.6 does NOT support `temperature=0`.
We omit the temperature parameter entirely so it uses the model default.

Two LLM-based evaluators:
1. **Correctness** — Is the predicted answer factually equivalent to the reference?
2. **Conciseness** — Is the answer brief and to-the-point?

In [ ]:
# ========================================
# 6a. CORRECTNESS evaluator (LLM-as-a-Judge)
# ========================================
# NOTE: No temperature parameter — GPT-5.6 only supports default (1).
# The strict grading prompt ensures deterministic-like behavior.

CORRECTNESS_SYSTEM = (
    "You are a strict grading assistant for a food safety and recipe Q&A bot.\n\n"
    "Your ONLY job: compare the predicted answer to the reference answer "
    "and decide if they are factually equivalent.\n\n"
    "Rules:\n"
    "- The predicted answer does NOT need to match word-for-word. "
    "Equivalent meaning counts as correct.\n"
    "- Minor rounding differences are acceptable "
    "(e.g. 28 g vs 28.35 g is CORRECT).\n"
    "- If the predicted answer contains the correct fact but adds extra "
    "safe context, that is CORRECT.\n"
    "- If the predicted answer is factually wrong, incomplete in a way "
    "that changes meaning, or contradicts the reference, that is INCORRECT.\n\n"
    "Respond with EXACTLY one word: CORRECT or INCORRECT\n"
    "No explanation. No punctuation. Just the single word."
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = (
        f"Question: {inputs['question']}\n"
        f"Reference answer: {reference_outputs['answer']}\n"
        f"Predicted answer: {outputs['response']}\n\n"
        f"Grade:"
    )

    response = openai_client.chat.completions.create(
        model="gpt-5.6",
        messages=[
            {"role": "system", "content": CORRECTNESS_SYSTEM},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content.strip()

    return response == "CORRECT"

print("Correctness evaluator defined (GPT-5.6).")

Correctness evaluator defined (GPT-5.6).


In [ ]:
# ========================================
# 6b. CONCISENESS evaluator (LLM-as-a-Judge)
# ========================================
# Upgraded from simple length check to LLM judge.

CONCISENESS_SYSTEM = (
    "You are a conciseness grading assistant.\n\n"
    "Your job: decide whether the predicted answer is concise — short, "
    "direct, and free of unnecessary filler — while still containing "
    "all essential information.\n\n"
    "Rules:\n"
    "- A one-sentence answer that covers the key fact is CONCISE.\n"
    "- A short answer (1-3 sentences) that stays on topic is CONCISE.\n"
    "- An answer that rambles, repeats itself, adds unnecessary disclaimers, "
    "or is significantly longer than needed is NOT_CONCISE.\n"
    "- Compare the predicted answer length and density against the reference. "
    "If the predicted answer is more than roughly 3x the length of the "
    "reference without adding essential new information, it is NOT_CONCISE.\n\n"
    "Respond with EXACTLY one word: CONCISE or NOT_CONCISE\n"
    "No explanation. No punctuation. Just the word(s)."
)

def conciseness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = (
        f"Question: {inputs['question']}\n"
        f"Reference answer: {reference_outputs['answer']}\n"
        f"Predicted answer: {outputs['response']}\n\n"
        f"Grade:"
    )

    response = openai_client.chat.completions.create(
        model="gpt-5.6",
        messages=[
            {"role": "system", "content": CONCISENESS_SYSTEM},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content.strip()

    return response == "CONCISE"

print("Conciseness evaluator defined (GPT-5.6).")

Conciseness evaluator defined (GPT-5.6).


## 🤖 Define Target App (Recipe & FDA Safety Bot)

In [ ]:
# ========================================
# 7. Target application — the bot being evaluated
# ========================================

SYSTEM_PROMPT = (
    "You are a concise food safety and recipe assistant for a "
    "US restaurant/cloud kitchen. Provide practical recipe, prep, "
    "storage, reheating, allergen, and handling guidance aligned with "
    "the FDA Food Code and standard US food-safety practices.\n\n"
    "Answer in ONE sentence. Keep it crisp and operational.\n\n"
    "Safe defaults:\n"
    "- Cold holding: 41°F / 5°C or below\n"
    "- Hot holding: 135°F / 57°C or above\n"
    "- Reheat leftovers/TCS: 165°F / 74°C\n"
    "- Poultry: 165°F / 74°C\n"
    "- Ground meats: 160°F / 71°C\n"
    "- Whole cuts/fish: 145°F / 63°C\n"
    "- Cooling: 135°F to 70°F in 2 hrs, then to 41°F within 6 hrs total\n\n"
    "Major US allergens (Big Nine): milk, eggs, fish, crustacean shellfish, "
    "tree nuts, peanuts, wheat, soybeans, sesame.\n\n"
    "Do not claim FDA approval. Say: 'This is FDA Food Code-aligned "
    "guidance; confirm with your local health department.'\n"
    "If info is missing, ask one focused question."
)

def my_app(question, model="gpt-5.6", instructions=SYSTEM_PROMPT):
    """Call the target model. No temperature param for GPT-5.6."""
    return openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

print("Target app defined — default model: gpt-5.6")

Target app defined — default model: gpt-5.6


## 🚀 Run Evaluation — GPT-5.6

In [ ]:
# ========================================
# 8. Run evaluation — GPT-5.6
# ========================================

def ls_target(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"], model="gpt-5.6")}

experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, conciseness],
    experiment_prefix="gpt-5.6",
)

View the evaluation results for experiment: 'gpt-5.6-7fe37995' at:
https://smith.langchain.com/o/c8f8810e-4941-552b-aef3-15ad938ead98/datasets/1e2bbcdb-9d50-4c1d-8d28-e18c926554ad/compare?selectedSessions=5274e94f-186d-40cf-8726-96b45225995d




0it [00:00, ?it/s]

## 📊 (Optional) Compare with other models

Uncomment and run the cells below to benchmark GPT-5.6 against older models.

In [ ]:
# ========================================
# 9a. (Optional) Compare — GPT-5.1
# ========================================

# def ls_target_5_1(inputs: dict) -> dict:
#     return {"response": my_app(inputs["question"], model="gpt-5.1-2025-11-13")}
#
# experiment_results_5_1 = client.evaluate(
#     ls_target_5_1,
#     data=dataset_name,
#     evaluators=[correctness, conciseness],
#     experiment_prefix="gpt-5.1",
# )

In [ ]:
# ========================================
# 9b. (Optional) Compare — GPT-4.1 Nano
# ========================================

# def ls_target_nano(inputs: dict) -> dict:
#     return {"response": my_app(inputs["question"], model="gpt-4.1-nano-2025-04-14")}
#
# experiment_results_nano = client.evaluate(
#     ls_target_nano,
#     data=dataset_name,
#     evaluators=[correctness, conciseness],
#     experiment_prefix="gpt-4.1-nano",
# )

## ✅ Summary

**What this notebook does:**

1. Creates a 34-example dataset of recipe & FDA food-safety Q&A pairs
2. Runs a GPT-5.6 powered bot against all 34 questions
3. Uses **two LLM-as-a-Judge evaluators** (both GPT-5.6):
   - **Correctness** — factually equivalent to the reference?
   - **Conciseness** — brief and on-point?
4. Logs everything to LangSmith for comparison

**Key fix:** GPT-5.6 does not support `temperature=0`.
The temperature parameter is omitted from all API calls.

**View results:** LangSmith dashboard → Datasets → *Recipe Bot — FDA Safety Q/A — Cohort 11*